### BM25

In [1]:
from pathlib import Path
import json
import re
import unicodedata

from rank_bm25 import BM25Okapi

def normalize_text(text: str) -> str:
    """
    Normaliza o texto para busca BM25.

    - converte para minúsculas
    - remove acentos
    - mantém apenas palavras/números
    """
    text = text.lower()

    # Remove acentos
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    return text


def tokenize(text: str) -> list[str]:
    """
    Converte um texto em tokens para o BM25.
    """
    text = normalize_text(text)

    return re.findall(r"\b\w+\b", text)


def load_chunks(chunks_dir: Path) -> list[dict]:
    """
    Carrega todos os chunks dos arquivos JSONL.
    """

    chunks = []

    for jsonl_path in chunks_dir.glob("*.jsonl"):
        print(f"Carregando: {jsonl_path.name}")

        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                chunk = json.loads(line)
                chunks.append(chunk)

    return chunks


def build_bm25(chunks: list[dict]):
    """
    Cria o índice BM25 a partir dos chunks.
    """

    corpus = [
        tokenize(chunk["text"])
        for chunk in chunks
    ]

    return BM25Okapi(corpus)


def search_bm25(
    bm25,
    chunks: list[dict],
    query: str,
    top_k: int = 5,
):
    """
    Executa uma busca BM25 e retorna os chunks mais relevantes.
    """

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    # Índices ordenados pela maior pontuação
    ranked_indexes = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )

    results = []

    for index in ranked_indexes[:top_k]:
        chunk = chunks[index].copy()
        chunk["score"] = float(scores[index])

        results.append(chunk)

    return results

### BGE-M3

In [2]:
import numpy as np

def search_embeddings(
    model,
    embeddings,
    chunks,
    query,
    top_k=20,
):
    query_embedding = model.encode(
        query,
        normalize_embeddings=True,
    )

    scores = embeddings @ query_embedding

    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in indices:
        chunk = chunks[idx].copy()
        chunk["score"] = float(scores[idx])
        results.append(chunk)

    return results

### Fusion

In [12]:
from collections import defaultdict

def reciprocal_rank_fusion(
    rankings,
    k=60,
):
    scores = defaultdict(float)
    documents = {}

    for ranking in rankings:

        for rank, doc in enumerate(ranking, start=1):

            key = (
                doc["chunk_id"],
            )

            documents[key] = doc

            scores[key] += 1 / (k + rank)

    fused = []

    for key, score in scores.items():

        doc = documents[key].copy()
        doc["rrf_score"] = score

        fused.append(doc)

    fused.sort(
        key=lambda x: x["rrf_score"],
        reverse=True,
    )

    return fused

In [13]:
from pathlib import Path
import json
from sentence_transformers import SentenceTransformer

chunks_dir = Path("data/chunks")

# 1. Carrega todos os chunks
chunks = load_chunks(chunks_dir)

print()
print(f"Total de chunks: {len(chunks)}")

# 2. Cria índice BM25
bm25 = build_bm25(chunks)

print("Índice BM25 criado.")

# 3. Teste
query = "quais são as formas de pagamento?"

bm25_results = search_bm25(
    bm25,
    chunks,
    query,
    top_k=5,
)

model = SentenceTransformer("BAAI/bge-m3")

embeddings = np.load(
    "data/embeddings.npy"
)

embedding_results = search_embeddings(
    model,
    embeddings,
    chunks,
    query,
    top_k=5,
)

results = reciprocal_rank_fusion(
    [
        embedding_results,
        bm25_results,
    ]
)

Carregando: politicas_da_loja.jsonl

Total de chunks: 8
Índice BM25 criado.


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32416.94it/s]


In [14]:
for result in bm25_results:
        print(
            f"\nScore: {result['score']:.4f}"
        )
        print(
            f"Documento: {result['source']}"
        )
        print(
            f"Página: {result['page']}"
        )
        print(
            f"Chunk: {result['chunk_id']}"
        )
print("\n\n--- Resultados da embedding ---")
for result in embedding_results:
        print(
            f"\nScore: {result['score']:.4f}"
        )
        print(
            f"Documento: {result['source']}"
        )
        print(
            f"Página: {result['page']}"
        )
        print(
            f"Chunk: {result['chunk_id']}"
        )
print("\n\n--- Resultados da fusão ---")
for result in results:
        print(
            f"\nScore: {result['score']:.4f}"
        )
        print(
            f"Documento: {result['source']}"
        )
        print(
            f"Página: {result['page']}"
        )
        print(
            f"Chunk: {result['chunk_id']}"
        )


Score: 6.1985
Documento: politicas_da_loja.pdf
Página: 3
Chunk: politicas_da_loja_p3

Score: 1.7417
Documento: politicas_da_loja.pdf
Página: 4
Chunk: politicas_da_loja_p4

Score: 1.0810
Documento: politicas_da_loja.pdf
Página: 6
Chunk: politicas_da_loja_p6

Score: 1.0724
Documento: politicas_da_loja.pdf
Página: 2
Chunk: politicas_da_loja_p2

Score: 1.0620
Documento: politicas_da_loja.pdf
Página: 7
Chunk: politicas_da_loja_p7


--- Resultados da embedding ---

Score: 0.6541
Documento: politicas_da_loja.pdf
Página: 3
Chunk: politicas_da_loja_p3

Score: 0.5193
Documento: politicas_da_loja.pdf
Página: 6
Chunk: politicas_da_loja_p6

Score: 0.5093
Documento: politicas_da_loja.pdf
Página: 4
Chunk: politicas_da_loja_p4

Score: 0.4976
Documento: politicas_da_loja.pdf
Página: 7
Chunk: politicas_da_loja_p7

Score: 0.4971
Documento: politicas_da_loja.pdf
Página: 5
Chunk: politicas_da_loja_p5


--- Resultados da fusão ---

Score: 6.1985
Documento: politicas_da_loja.pdf
Página: 3
Chunk: politicas_d